# HostWise — Track A: Preprocessing Pipeline
## Setup

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.preprocessing import FunctionTransformer


In [ ]:

DATA_PATH = Path("../data/processed/hostwise_5cities.csv")
SPLIT_DIR = Path("../data/splits")

NUMERIC_FEATURES = ["accommodates", "bedrooms", "bathrooms", "latitude", "longitude"]
CATEGORICAL_FEATURES = ["city", "room_type", "property_type"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "price_num"

## Load the shared train/test data
Rebuild the exact train/test rows from the saved listing IDs (no new split).

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False, dtype={"id": "int64"}).set_index("id")
df[TARGET] = pd.to_numeric(df["price"].str.replace(r"[$,]", "", regex=True), errors="coerce")

train_ids = pd.read_csv(SPLIT_DIR / "train_ids.csv", dtype={"id": "int64"})["id"]
test_ids = pd.read_csv(SPLIT_DIR / "test_ids.csv", dtype={"id": "int64"})["id"]

X_train, y_train = df.loc[train_ids, FEATURES], df.loc[train_ids, TARGET]
X_test, y_test = df.loc[test_ids, FEATURES], df.loc[test_ids, TARGET]

X_train.shape, X_test.shape, y_train.isna().sum()

((123049, 8), (30763, 8), 0)

## Preprocessing steps
Numerical: median imputation, then standard scaling. Categorical: most-frequent imputation, then one-hot encoding.

In [3]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

## Preprocessor (ColumnTransformer)
Applies the numerical pipeline to numerical features and the categorical pipeline to categorical features.

In [4]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_FEATURES),
    ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
])
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['accommodates', 'bedrooms', 'bathrooms',
                                  'latitude', 'longitude']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['city', 'room_type', 'property_type'])])

## Validate the preprocessor
Fit on the training data only, then transform the test data with the learned parameters.

In [5]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)


In [6]:
X_train_prep.shape, X_test_prep.shape, pd.isna(X_train_prep).sum(), pd.isna(X_test_prep).sum()

((123049, 111), (30763, 111), 0, 0)

## Inspect the transformed features

In [7]:
pd.DataFrame(X_train_prep, columns=preprocessor.get_feature_names_out(), index=X_train.index).head()

,num__accommodates,num__bedrooms,num__bathrooms,num__latitude,num__longitude,cat__city_barcelona,cat__city_london,cat__city_madrid,cat__city_paris,cat__city_rome,...,cat__property_type_Shared room in hotel,cat__property_type_Shared room in loft,cat__property_type_Shared room in rental unit,cat__property_type_Shared room in townhouse,cat__property_type_Shared room in villa,cat__property_type_Shepherd’s hut,cat__property_type_Tent,cat__property_type_Tiny home,cat__property_type_Tower,cat__property_type_Yurt
id,,,,,,,,,,,,,,,,,,,,,
1339822709246176705,0.627701,0.433125,-0.437894,0.380413,-0.083260,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1638177765178487644,-0.773620,-0.567600,-0.437894,0.961413,-0.573813,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12896531,2.496129,1.433850,0.977774,-1.174515,1.869607,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1262318958906657660,-0.773620,-0.567600,-0.437894,0.969034,-0.561743,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1649222827954135613,2.029022,2.434575,3.101278,0.956712,-0.568830,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Check compatibility with a model pipeline and cross-validation
A placeholder model (DummyRegressor) confirms that the preprocessor can be refitted inside each cross-validation fold. This is a compatibility check, not model training.

In [8]:
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_score

check_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", DummyRegressor(strategy="median")),
])

cross_val_score(check_pipeline, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")

array([-158.99827509, -164.36026453, -155.33815441, -160.75903373,
       -156.82120931])

## Save the preprocessor for Track B
The unfitted preprocessor is saved so Track B uses exactly the same preprocessing definition.

In [9]:
import joblib

Path("../models").mkdir(exist_ok=True)
joblib.dump(clone(preprocessor), "../models/preprocessor_v1.joblib")

['../models/preprocessor_v1.joblib']

## Preprocessing V2
### Feature groups and V2 training data

In [10]:
NUMERIC_V2 = ["accommodates", "bathrooms", "latitude", "longitude", "beds", "minimum_nights", "hosts_time_as_host_years"]
NUMERIC_FLAG_V2 = ["bedrooms", "review_scores_rating"]
LOG_NUMERIC_V2 = ["number_of_reviews", "calculated_host_listings_count"]
CATEGORICAL_V2 = ["city", "room_type", "property_type", "neighbourhood_cleansed", "host_is_superhost"]
FEATURES_V2 = NUMERIC_V2 + NUMERIC_FLAG_V2 + LOG_NUMERIC_V2 + CATEGORICAL_V2

X_train_v2 = df.loc[train_ids, FEATURES_V2]
X_test_v2 = df.loc[test_ids, FEATURES_V2]

X_train_v2.shape, X_test_v2.shape

((123049, 16), (30763, 16))

### Numeric branches

In [14]:
numeric_flag_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

log_numeric_pipeline = Pipeline([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

### Categorical branch and V2 preprocessor

In [15]:
categorical_grouped_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist", sparse_output=False)),
])

preprocessor_v2 = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_V2),
    ("num_flag", numeric_flag_pipeline, NUMERIC_FLAG_V2),
    ("num_log", log_numeric_pipeline, LOG_NUMERIC_V2),
    ("cat", categorical_grouped_pipeline, CATEGORICAL_V2),
])

preprocessor_v2

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['accommodates', 'bathrooms', 'latitude',
                                  'longitude', 'beds', 'minimum_nights',
                                  'hosts_time_as_host_years']),
                                ('num_flag',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scaler', StandardScaler...
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['number_of_reviews',
                                  'calculated_host_listings_count']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=50,
                                                                sparse_output=False))]),
                                 ['city', 'room_type', 'property_type',
                                  'neighbourhood_cleansed',
                                  'host_is_superhost'])])

### Validate V2 (fit on training data only)

In [16]:
X_train_v2_prep = preprocessor_v2.fit_transform(X_train_v2)
X_test_v2_prep = preprocessor_v2.transform(X_test_v2)

X_train_v2_prep.shape, X_test_v2_prep.shape, pd.isna(X_train_v2_prep).sum(), pd.isna(X_test_v2_prep).sum()

((123049, 214), (30763, 214), 0, 0)

### Inspect category grouping

In [17]:
encoder_v2 = preprocessor_v2.named_transformers_["cat"].named_steps["encoder"]

pd.DataFrame({
    "categories_in_train": [len(c) for c in encoder_v2.categories_],
    "grouped_as_infrequent": [0 if c is None else len(c) for c in encoder_v2.infrequent_categories_],
}, index=CATEGORICAL_V2)

,categories_in_train,grouped_as_infrequent
city,5,0
room_type,4,0
property_type,97,62
neighbourhood_cleansed,259,106
host_is_superhost,2,0


### Check V2 inside a cross-validated pipeline

In [18]:
check_pipeline_v2 = Pipeline([
    ("preprocessor", clone(preprocessor_v2)),
    ("model", DummyRegressor(strategy="median")),
])

cross_val_score(check_pipeline_v2, X_train_v2, y_train, cv=5, scoring="neg_mean_absolute_error")

array([-158.99827509, -164.36026453, -155.33815441, -160.75903373,
       -156.82120931])

### Shared preprocessors from preprocessing.py

In [19]:
from preprocessing import preprocessor_v2 as shared_preprocessor_v2

shared_preprocessor_v2

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['accommodates', 'bathrooms', 'latitude',
                                  'longitude', 'beds', 'minimum_nights',
                                  'hosts_time_as_host_years']),
                                ('num_flag',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scaler', StandardScaler...
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['number_of_reviews',
                                  'calculated_host_listings_count']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=50,
                                                                sparse_output=False))]),
                                 ['city', 'room_type', 'property_type',
                                  'neighbourhood_cleansed',
                                  'host_is_superhost'])])

## V2.1 proposal: bathrooms from `bathrooms_text`
### Check recovery and agreement on training data

In [20]:
bath_text = X_train_v2.join(df["bathrooms_text"])["bathrooms_text"].str.lower()
bath_count_text = bath_text.str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
bath_count_text = bath_count_text.mask(bath_text.str.contains("half-bath", na=False), 0.5)

bathrooms = X_train_v2["bathrooms"]
pd.Series({
    "bathrooms missing": bathrooms.isna().sum(),
    "recoverable from text": (bathrooms.isna() & bath_count_text.notna()).sum(),
    "still missing": (bathrooms.isna() & bath_count_text.isna()).sum(),
    "both present, values differ": (bathrooms.notna() & bath_count_text.notna() & (bathrooms != bath_count_text)).sum(),
}).to_frame("training listings")

,training listings
bathrooms missing,7838
recoverable from text,7719
still missing,119
"both present, values differ",36


### Inspect the 36 disagreements

In [21]:
differs = bathrooms.notna() & bath_count_text.notna() & (bathrooms != bath_count_text)

pd.DataFrame({
    "bathrooms_text": bath_text[differs],
    "bathrooms": bathrooms[differs],
    "parsed": bath_count_text[differs],
}).value_counts().to_frame("listings")

listings
bathrooms_text bathrooms parsed          
2 baths        1.0       2.0            7
0 baths        1.0       0.0            4
1 bath         1.5       1.0            4
               2.0       1.0            3
2 baths        3.0       2.0            3
1.5 baths      1.0       1.5            2
2.5 baths      2.0       2.5            1
3.5 baths      2.5       3.5            1
3 baths        4.0       3.0            1
               2.5       3.0            1
2.5 baths      3.5       2.5            1
               3.0       2.5            1
2 baths        2.5       2.0            1
2.5 baths      1.5       2.5            1
0 baths        2.0       0.0            1
1 private bath 2.0       1.0            1
0 shared baths 1.0       0.0            1
               0.5       0.0            1
4 baths        3.5       4.0            1

## Preprocessing V2.1: bathrooms from `bathrooms_text`
### Parsing function

In [22]:
def parse_bathrooms(X):
    text = X.iloc[:, 0].astype("string").str.lower()
    count = text.str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
    count = count.mask(text.str.contains("half-bath", na=False), 0.5)
    return pd.DataFrame({
        "bath_count": count,
        "bath_shared": text.str.contains("shared", na=False).astype(int),
        "bath_private": text.str.contains("private", na=False).astype(int),
    }, index=X.index)


def bathroom_feature_names(transformer, input_features):
    return ["bath_count", "bath_shared", "bath_private"]


parse_bathrooms(df.loc[train_ids, ["bathrooms_text"]]).describe()

,bath_count,bath_shared,bath_private
count,122899.000000,123049.000000,123049.000000
mean,1.315523,0.106527,0.098132
std,0.728595,0.308512,0.297494
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000
75%,1.500000,0.000000,0.000000
max,38.000000,1.000000,1.000000


### V2.1 preprocessor
`bathrooms` is replaced by three features parsed from `bathrooms_text` inside the pipeline.

In [23]:
NUMERIC_V21 = [f for f in NUMERIC_V2 if f != "bathrooms"]
BATHROOM_V21 = ["bathrooms_text"]
FEATURES_V21 = NUMERIC_V21 + NUMERIC_FLAG_V2 + LOG_NUMERIC_V2 + BATHROOM_V21 + CATEGORICAL_V2

bathroom_pipeline = Pipeline([
    ("parse", FunctionTransformer(parse_bathrooms, feature_names_out=bathroom_feature_names)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor_v21 = ColumnTransformer([
    ("num", numeric_pipeline, NUMERIC_V21),
    ("num_flag", numeric_flag_pipeline, NUMERIC_FLAG_V2),
    ("num_log", log_numeric_pipeline, LOG_NUMERIC_V2),
    ("bath", bathroom_pipeline, BATHROOM_V21),
    ("cat", categorical_grouped_pipeline, CATEGORICAL_V2),
])

preprocessor_v21

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['accommodates', 'latitude', 'longitude',
                                  'beds', 'minimum_nights',
                                  'hosts_time_as_host_years']),
                                ('num_flag',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['bedr...
                                                 ('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['bathrooms_text']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=50,
                                                                sparse_output=False))]),
                                 ['city', 'room_type', 'property_type',
                                  'neighbourhood_cleansed',
                                  'host_is_superhost'])])

### Validate V2.1 (fit on training data only)

In [24]:
X_train_v21 = df.loc[train_ids, FEATURES_V21]
X_test_v21 = df.loc[test_ids, FEATURES_V21]

X_train_v21_prep = preprocessor_v21.fit_transform(X_train_v21)
X_test_v21_prep = preprocessor_v21.transform(X_test_v21)

X_train_v21_prep.shape, X_test_v21_prep.shape, pd.isna(X_train_v21_prep).sum(), pd.isna(X_test_v21_prep).sum()

((123049, 216), (30763, 216), 0, 0)

### Check V2.1 on a new, unseen listing
A brand-new listing: no reviews, missing bedrooms and bathroom text, and a neighbourhood and property type never seen in training.

In [25]:
new_listing = pd.DataFrame([{
    "accommodates": 2, "latitude": 51.51, "longitude": -0.13, "beds": 1, "minimum_nights": 2,
    "hosts_time_as_host_years": 0, "bedrooms": np.nan, "review_scores_rating": np.nan,
    "number_of_reviews": 0, "calculated_host_listings_count": 1, "bathrooms_text": np.nan,
    "city": "london", "room_type": "Entire home/apt", "property_type": "Unseen property type",
    "neighbourhood_cleansed": "Unseen neighbourhood", "host_is_superhost": "f",
}])[FEATURES_V21]

new_listing_prep = preprocessor_v21.transform(new_listing)
new_listing_prep.shape, pd.isna(new_listing_prep).sum()

((1, 216), 0)

### Check V2.1 inside a cross-validated pipeline

In [26]:
check_pipeline_v21 = Pipeline([
    ("preprocessor", clone(preprocessor_v21)),
    ("model", DummyRegressor(strategy="median")),
])

cross_val_score(check_pipeline_v21, X_train_v21, y_train, cv=5, scoring="neg_mean_absolute_error")

array([-158.99827509, -164.36026453, -155.33815441, -160.75903373,
       -156.82120931])

### Shared V2.1 preprocessor from preprocessing.py

In [27]:
import importlib
import preprocessing

importlib.reload(preprocessing)
preprocessing.preprocessor_v21.fit_transform(X_train_v21).shape

(123049, 216)

## Summary
- **V1** (8 features → 111 columns): morning baseline, unchanged.
- **V2** (16 features → 214 columns): adds neighbourhood, reviews (rating with missing flag, log review count), host (superhost, log listings count, tenure), beds and minimum nights; rare categories grouped (`min_frequency=50`).
- **V2.1** (16 features → 216 columns): V2 with `bathrooms` replaced by `bath_count`, `bath_shared` and `bath_private` parsed from `bathrooms_text`. Recovers 7,719 of 7,838 missing training counts (98.5%); text and numeric values agree on 99.97% of listings where both exist.
- **Leakage safety:** all learned steps are fitted on training data only and refitted per cross-validation fold; parsing and log transforms are stateless.
- **Open items:** "0 baths" entries that may mean "not filled in"; capping extreme size values; tuning `min_frequency`; distance to centre and amenity count (deferred).
- **Next:** Track B compares V1, V2 and V2.1 with the same model and cross-validation folds on the training data.